In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
df = pd.read_csv("train.csv")

print(df.shape)
print(df.head())

In [ ]:
print(df.info())

In [ ]:
print(df.describe())

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)

print(missing[missing > 0])
print("Total missing values:", df.isna().sum().sum())

In [ ]:
X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
train_missing = X_train.isna().sum().sort_values(ascending=False)
test_missing = X_test.isna().sum().sort_values(ascending=False)

print("TRAIN missing:")
print(train_missing[train_missing > 0])

print("\nTEST missing:")
print(test_missing[test_missing > 0])

In [ ]:
#Check number of categorical and numericals columns
print(df.dtypes.value_counts())

In [ ]:
#Create our categorical and numerical column lists after the split:
num_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

cat_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [ ]:
#two alternative ways of defining the imputer.
#Option1 
numeric_transformer = SimpleImputer(
    strategy="median"
)

In [ ]:
#two alternative ways of defining the imputer.
#Option1 
categorical_imputer = SimpleImputer(
    strategy="constant",
    fill_value="None"
)

In [ ]:
absence_cat_cols = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "MasVnrType",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2"
]

other_cat_cols = [
    col for col in cat_cols
    if col not in absence_cat_cols
]

print("Absence-based categorical:", len(absence_cat_cols))
print("Other categorical:", len(other_cat_cols))
print("Numerical:", len(num_cols))

In [ ]:
#two alternative ways of defining the imputer.
#Option 2
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

In [ ]:
#two alternative ways of defining the imputer.
#Option 2 
absence_cat_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value="None"
        )),
        ("encoder", OneHotEncoder(
            drop="first",
            handle_unknown="ignore"
        ))
    ]
)

In [ ]:
#two alternative ways of defining the imputer.
#Option 2 
other_cat_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(
            strategy="most_frequent"
        )),
        ("encoder", OneHotEncoder(
            drop="first",
            handle_unknown="ignore"
        ))
    ]
)

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("absence_cat", absence_cat_transformer, absence_cat_cols),
        ("other_cat", other_cat_transformer, other_cat_cols)
    ]
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [ ]:
print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

In [ ]:
print("Missing values in X_train:",
      np.isnan(X_train_processed.toarray()).sum())

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

In [ ]:
xgb_model.fit(
    X_train_processed,
    y_train
)

In [ ]:
y_train_pred_xgb = xgb_model.predict(
    X_train_processed
)

y_test_pred_xgb = xgb_model.predict(
    X_test_processed
)

In [ ]:
train_r2_xgb = r2_score(
    y_train,
    y_train_pred_xgb
)

test_r2_xgb = r2_score(
    y_test,
    y_test_pred_xgb
)

print("XGBoost Training R²:", train_r2_xgb)
print("XGBoost Test R²:", test_r2_xgb)

In [ ]:
n_estimators_values = [50, 100, 200, 300]
learning_rates = [0.05, 0.1]

for lr in learning_rates:

    for n in n_estimators_values:

        xgb = XGBRegressor(
            n_estimators=n,
            learning_rate=lr,
            max_depth=3,
            random_state=42
        )

        xgb.fit(
            X_train_processed,
            y_train
        )

        train_pred = xgb.predict(X_train_processed)
        test_pred = xgb.predict(X_test_processed)

        train_r2 = r2_score(y_train, train_pred)
        test_r2 = r2_score(y_test, test_pred)

        print(
            f"Learning Rate: {lr} | "
            f"Trees: {n} | "
            f"Train R²: {train_r2:.4f} | "
            f"Test R²: {test_r2:.4f}"
        )

In [ ]:
#investigate max_depth + min_child_weight.

max_depth_values = [2, 3, 4, 5]
min_child_weight_values = [1, 3, 5]

for depth in max_depth_values:

    for child_weight in min_child_weight_values:

        xgb = XGBRegressor(
            n_estimators=300,
            learning_rate=0.1,
            max_depth=depth,
            min_child_weight=child_weight,
            random_state=42
        )

        xgb.fit(
            X_train_processed,
            y_train
        )

        train_pred = xgb.predict(X_train_processed)
        test_pred = xgb.predict(X_test_processed)

        train_r2 = r2_score(y_train, train_pred)
        test_r2 = r2_score(y_test, test_pred)

        print(
            f"Depth: {depth} | "
            f"Min Child Weight: {child_weight} | "
            f"Train R²: {train_r2:.4f} | "
            f"Test R²: {test_r2:.4f}"
        )

In [ ]:
#Investigate with subsample and colsample
subsample_values = [0.7, 0.8, 1.0]
colsample_values = [0.7, 0.8, 1.0]

for subsample in subsample_values:

    for colsample in colsample_values:

        xgb = XGBRegressor(
            n_estimators=300,
            learning_rate=0.1,
            max_depth=3,
            min_child_weight=5,
            subsample=subsample,
            colsample_bytree=colsample,
            random_state=42
        )

        xgb.fit(
            X_train_processed,
            y_train
        )

        train_pred = xgb.predict(X_train_processed)
        test_pred = xgb.predict(X_test_processed)

        train_r2 = r2_score(y_train, train_pred)
        test_r2 = r2_score(y_test, test_pred)

        print(
            f"Subsample: {subsample} | "
            f"Colsample: {colsample} | "
            f"Train R²: {train_r2:.4f} | "
            f"Test R²: {test_r2:.4f}"
        )

In [ ]:
param_grid_xgb = {
    "n_estimators": [200, 300],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 4],
    "min_child_weight": [3, 5],
    "subsample": [0.7, 0.8],
    "colsample_bytree": [0.8, 1.0],
    "reg_alpha": [0, 0.1],
    "reg_lambda": [1, 5]
}

In [ ]:
xgb = XGBRegressor(
    random_state=42
)

In [ ]:
from sklearn.model_selection import GridSearchCV
grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid_xgb,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

In [ ]:
grid_xgb.fit(
    X_train_processed,
    y_train
)

In [ ]:
print("Best Parameters:", grid_xgb.best_params_)
print("Best CV R²:", grid_xgb.best_score_)

In [ ]:
best_xgb = grid_xgb.best_estimator_

In [ ]:
y_test_pred_xgb = best_xgb.predict(
    X_test_processed
)

final_xgb_r2 = r2_score(
    y_test,
    y_test_pred_xgb
)

print("Final XGBoost Test R²:", final_xgb_r2)